# Tutorial: Q2–Q4 in Python

Worked Python answers for the Week 01 tutorial. Q2 is a service centre with a single server, Q3 is a workshop that
works in 15-minute periods, and Q4 is an n-sided die. They follow the Excel layout of `Tutorial Answers.xlsx`
column by column: draw a random number, look it up in the cumulative-probability table, then apply the same formulas.

*Made for this site: the tutorial's Q4b asks for this code. Uses only the standard library, NumPy and pandas, so there is
nothing to install.*

In [1]:
import random
import numpy as np
import pandas as pd

rng = random.Random(1)   # fixed seed: same tables every run; change it to see other runs

## Sampling from a table (the Excel `VLOOKUP(RAND(), …)` trick)

Build the cumulative probabilities (0, 0.1, 0.3, …). Draw u uniformly from [0, 1). The sampled value is the last one
whose cumulative start is ≤ u. `rng.choices` does the same thing in one line, and both are shown here.

In [2]:
def lookup(values, probs, u):
    """Excel-style lookup: the value whose cumulative-probability band contains u."""
    start = 0.0
    for v, p in zip(values, probs):
        if u < start + p:
            return v
        start += p
    return values[-1]

def draw(values, probs):
    return lookup(values, probs, rng.random())   # same as rng.choices(values, probs)[0]

## Q2: service centre, 30 customers

Inter-arrival times 5–30 minutes; service times 10–30 minutes. The expected values show why a queue builds up:
mean inter-arrival time 15.15 min, mean service time 14.5 min, so the server is busy about 96% of the time.
(In the original `Tutorial Answers.xlsx` the expected inter-arrival time left out the 25 and 30 values and showed 12.5;
this is corrected there too, erratum C4.)

In [3]:
IAT = ([5, 10, 15, 20, 25, 30], [0.10, 0.20, 0.40, 0.20, 0.07, 0.03])
SD = ([10, 12, 14, 16, 20, 30], [0.05, 0.20, 0.45, 0.20, 0.09, 0.01])
print(f"Expected inter-arrival time {np.dot(*IAT):.2f} min, expected service time {np.dot(*SD):.2f} min, "
      f"utilisation {np.dot(*SD) / np.dot(*IAT):.3f}")

rows, at, se = [], 0, 0
for n in range(1, 31):
    iat = draw(*IAT)
    at = at + iat                    # AT(n) = AT(n-1) + IAT(n)
    ss = max(at, se)                 # SS(n) = max(SE(n-1), AT(n))
    sd = draw(*SD)
    idle = ss - se                   # server waiting time SS(n) - SE(n-1)
    se = ss + sd                     # SE(n) = SS(n) + SD(n)
    rows.append(dict(Customer=n, IAT=iat, AT=at, SS=ss, SD=sd, SE=se, CustomerWait=ss - at, ServerWait=idle))
q2 = pd.DataFrame(rows).set_index("Customer")
q2

Expected inter-arrival time 15.15 min, expected service time 14.50 min, utilisation 0.957


,IAT,AT,SS,SD,SE,CustomerWait,ServerWait
Customer,,,,,,,
1,10,10,10,16,26,0,10
2,20,30,30,14,44,0,4
3,15,45,45,14,59,0,1
4,15,60,60,16,76,0,1
5,5,65,76,10,86,11,0
6,20,85,86,14,100,1,0
7,20,105,105,10,115,0,5
8,15,120,120,16,136,0,5
9,10,130,136,20,156,6,0


In [4]:
print(f"Average customer wait {q2.CustomerWait.mean():.1f} min (max {q2.CustomerWait.max()}), "
      f"customers who waited {100 * (q2.CustomerWait > 0).mean():.0f}%, server idle {q2.ServerWait.sum()} of {q2.SE.iloc[-1]} min")

Average customer wait 4.7 min (max 20), customers who waited 57%, server idle 62 of 476 min


**What it shows.** With utilisation near 1, waits tend to grow along the run. A second server, or faster service
(fewer 20- and 30-minute jobs), would reduce them. Change the seed and run again: the pattern holds, the numbers change.

## Q3: workshop, 16 periods of 15 minutes (4 hours)

Each period, jobs arrive (5, 7, 11 or 12) and the stage can process 6, 8, 10 or 15. Jobs not done wait for the next period.

In [5]:
JA = ([5, 7, 11, 12], [0.10, 0.20, 0.40, 0.30])
JK = ([6, 8, 10, 15], [0.15, 0.40, 0.35, 0.10])
print(f"Expected arrivals {np.dot(*JA):.1f} jobs/period, expected capacity {np.dot(*JK):.1f} jobs/period")

rows, left, done = [], 0, 0
for t in range(1, 17):
    ja = draw(*JA)
    jq = ja + left                   # JQ(t) = JA(t) + TJL(t-1)
    jk = draw(*JK)
    jc = min(jq, jk)                 # JC(t) = min(JQ(t), JK(t))
    done += jc
    left = jq - jc                   # TJL(t) = JQ(t) - JC(t)
    rows.append(dict(Period=f"{t} ({15*(t-1)}-{15*t})", JA=ja, JQ=jq, JK=jk, JC=jc, TJC=done, TJL=left, UnusedCapacity=jk - jc))
q3 = pd.DataFrame(rows).set_index("Period")
q3

Expected arrivals 9.9 jobs/period, expected capacity 9.1 jobs/period


,JA,JQ,JK,JC,TJC,TJL,UnusedCapacity
Period,,,,,,,
1 (0-15),7,7,8,7,7,0,1
2 (15-30),12,12,10,10,17,2,0
3 (30-45),11,13,8,8,25,5,0
4 (45-60),11,16,10,10,35,6,0
5 (60-75),11,17,8,8,43,9,0
6 (75-90),11,20,6,6,49,14,0
7 (90-105),5,19,10,10,59,9,0
8 (105-120),12,21,10,10,69,11,0
9 (120-135),11,22,8,8,77,14,0


**What it shows.** Arrivals (9.9 per period) exceed capacity (9.1) on average, so the backlog grows, even though some
capacity is wasted in quiet periods. More capacity, or smoothing the arrivals, is needed.

## Q4: an n-sided die

The observed frequency of each face approaches 1/n as the number of trials k grows.

In [6]:
def dice(n_sides=6, k=10_000, seed=1):
    r = np.random.default_rng(seed)
    rolls = r.integers(1, n_sides + 1, size=k)
    freq = np.bincount(rolls, minlength=n_sides + 1)[1:] / k
    return pd.Series(freq, index=range(1, n_sides + 1), name=f"observed (k={k})")

for k in (60, 600, 60_000):
    print(f"k = {k:>6}: P(1) observed {dice(6, k)[1]:.4f}   (theory {1/6:.4f})")
dice(8, 8000).to_frame().assign(theory=1/8)

k =     60: P(1) observed 0.2000   (theory 0.1667)
k =    600: P(1) observed 0.1817   (theory 0.1667)
k =  60000: P(1) observed 0.1688   (theory 0.1667)


,observed (k=8000),theory
1,0.129375,0.125
2,0.124125,0.125
3,0.122625,0.125
4,0.125750,0.125
5,0.122250,0.125
6,0.119500,0.125
7,0.127750,0.125
8,0.128625,0.125
